In [1]:
import os
from typing import Literal
from typing import TypedDict
from dotenv import load_dotenv

from langgraph.graph import StateGraph
from langgraph.graph import END
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import WikipediaQueryRun
from langchain.utilities import WikipediaAPIWrapper
from langchain_google_genai import ChatGoogleGenerativeAI

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load env
load_dotenv()

True

In [3]:
# Step 1: Define custom state class for retry & fallback
class AgentState(TypedDict):
    input: str
    selected_tool: Literal['wikipedia','tavily']
    output: str
    error: str
    retries: int

In [4]:
# Step 2: Setup LLM and Tools
llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro", temperature=0.9)

In [5]:
# Setup Tools
wikipedia_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
tavily_tool = TavilySearchResults()

In [6]:
MAX_RETRIES = 2  # Max allowed retry attempts

In [7]:
# Step 3: Tool Router Agent
def tool_router(state: AgentState) -> AgentState:
    query = state["input"]
    prompt = f"""
    Decide which tool is better for answering this input:
    - If general knowledge, history, people: choose 'wikipedia'
    - If trending, current or product queries: choose 'tavily'
    Just return: wikipedia or tavily.
    Input: {query}
    """
    choice = llm.invoke(prompt).content.strip().lower()
    tool = "wikipedia" if "wiki" in choice else "tavily"
    print(f"Tool selected: {tool}")
    state["selected_tool"] = tool
    return state

In [8]:
# Step 4: Wikipedia Tool Node with try/except
def wikipedia_tool_node(state: AgentState) -> AgentState:
    query = state["input"]
    print(f"Wikipedia searching: {query}")
    try:
        result = wikipedia_tool.run(query)
        state["output"] = result
        state["error"] = ""
    except Exception as e:
        state["error"] = str(e)
        state["output"] = ""
    return state

In [9]:
# Step 5: Tavily Tool Node with try/except
def tavily_tool_node(state: AgentState) -> AgentState:
    query = state["input"]
    print(f"Tavily searching: {query}")
    try:
        result = tavily_tool.run(query)
        if isinstance(result, list) and result:
            content = result[0].get("content", "")
            state["output"] = content[:1000]
        elif isinstance(result, str):
            state["output"] = result[:1000]
        else:
            state["output"] = "No content"
        state["error"] = ""
    except Exception as e:
        state["error"] = str(e)
        state["output"] = ""
    return state

In [10]:
# Step 6: Guardrail Node
def guardrail_node(state: AgentState) -> Literal["end", "retry", "fallback"]:
    if state["error"] == "":
        return "end"
    elif state["retries"] < MAX_RETRIES:
        print(f"Retry #{state['retries'] + 1}")
        state["retries"] += 1
        return "retry"
    else:
        print("Max retries exceeded. Going to fallback.")
        return "fallback"

In [11]:
# Step 7: Fallback Node
def fallback_node(state: AgentState) -> AgentState:
    print("Fallback node running: Returning default answer.")
    state["output"] = "Sorry, I couldn't find a confident answer."
    state["error"] = "fallback_used"
    return state

In [12]:
# Step 8: Build the graph
builder = StateGraph(AgentState)
builder.add_node('tool_router',tool_router)
builder.add_node('wikipedia_tool',wikipedia_tool_node)
builder.add_node('tavily_tool',tavily_tool_node)
builder.add_node('fallback_tool',fallback_node)

In [13]:
# Wikipedia path
builder.add_conditional_edges("wikipedia_tool", guardrail_node, {
    "retry": "wikipedia_tool",
    "fallback": "fallback_tool",
    "end": END
})

In [14]:
# Tavily path
builder.add_conditional_edges("tavily_tool", guardrail_node, {
    "retry": "tavily_tool",
    "fallback": "fallback_tool",
    "end": END
})

In [15]:
builder.set_entry_point('tool_router')
builder.add_conditional_edges("tool_router", lambda s: s["selected_tool"], {
    "wikipedia": "wikipedia_tool",
    "tavily": "tavily_tool"
})

In [16]:
builder.add_edge("fallback_tool", END)

In [17]:
# Step 9: Compile and Test
graph = builder.compile()

query = {
    "input": "ajsdfjkasdfjkajf",
    "selected_tool": "",
    "output": "",
    "error": "",
    "retries": 0
}

In [18]:
result = graph.invoke(query)
print("\nFinal Output:", result["output"])
print("Error Info:", result["error"])
print("Retry Count:", result["retries"])

Tool selected: tavily
Tavily searching: ajsdfjkasdfjkajf

Final Output: ... a s d f j k l ;). • Place your left index finger on F and your right index finger on J. • For more information, see the keyboarding chart on the next page.
Error Info: 
Retry Count: 0
